In [23]:
# Martin MacDonald
# Honours Project
# 2D Discrete Traffic Model

In [24]:
#Imports
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap
from queue import PriorityQueue
from PIL import Image, ImageDraw
import matplotlib.pyplot as plt
import io
from collections import deque

## Setup the grid

In [25]:
#Setup grid.
def setup_grid(num_cars, rows, columns):
    total_size = rows * columns
    grid = np.zeros((rows, columns), dtype=np.int16)
    
    #Add randomly generated cars to test.
    num_cars = num_cars
    
    #Ensure there are not more cars than the total size of the grid, throw a value error if there are.
    if num_cars > total_size:
        raise ValueError("There are too many cars for the size of the grid")
    
    #Define the Car class, used to hold data about each car.
    class Car:
        def __init__(self, car_id, position, goal):
            self.id = car_id
            self.position = position
            self.goal = goal
            self.path = []
            self.wait_counter = 0
            self.total_distance_travelled = 0
            self.steps_taken = 0
            self.reached_goal = False
    
    #Cars dictionary which holds each car, its ID, positon & goal.
    cars = {}
    
    #Generates a randomly positioned car, checks if that space is already occupied, if it is, try again, 
    #if not, add car to grid & dictionary and give it a unique ID.
    car_id = 1
    used_goals = set()
    
    while car_id <= num_cars:
        #Generate random position
        row = np.random.randint(0, rows)
        col = np.random.randint(0, columns)
        #Check if the car can spawn on unoccupied space.
        if grid[row, col] == 0:
            goal_row = np.random.randint(0, rows)
            goal_column = np.random.randint(0, columns)
            #Check if the newly created goal has been used before or if it is the same position as the newly generated car. If it is, re-generate a new goal until sufficient.
            while (goal_row, goal_column) in used_goals or (goal_row, goal_column) == (row, col):
                goal_row = np.random.randint(0, rows)
                goal_column = np.random.randint(0, columns)
            #Add that newly created goal to used_goals
            used_goals.add((goal_row, goal_column))
            #If the space is clear, add that car to cars dictionary & grid and increment car_id by 1.
            grid[row, col] = car_id
            cars[car_id] = Car(car_id, (row, col), (goal_row, goal_column))
            car_id += 1

    return grid, cars

## Heuristic

In [26]:
#Manhattan Distance
def manhattan_distance(position_1, position_2):
    return (abs(position_1[0] - position_2[0]) + abs(position_1[1] - position_2[1]))

## Get Neighbours

In [27]:
#Get neighbours for a given position.
def get_neighbours(position, rows, columns):
    #Get the current cars position and split it into row and column.
    row, column = position
    #Setup a list to contain neighbours.
    neighbours = []

    #Create a list of tuples which represent each cardinal direction/possible move
    directions = [(-1, 0), (1, 0), (0, -1), (0, 1)]

    #For each direction, get the new position when taking that direction into account. 
    #If that position would be off the side of the grid, don't append it to the neighbours list.
    for delta_row, delta_column in directions:
        new_row, new_column = row + delta_row, column + delta_column

        if 0 <= new_row < rows and 0 <= new_column < columns:
            neighbours.append((new_row, new_column))

    #Return all the valid neighbours of the current car. 
    return neighbours

## Reconstruct path

In [28]:
#Reconstruct the path from start to goal using came_from.
def reconstruct_path(came_from, start, goal):
    #Set the current position to goal.
    current = goal
    #Setup a list to contain the path.
    path = []

    #While current has not reached the start, add the current step to the path 
    #then set current to its parent step (the next step along on the way to start).
    while current != start:
        path.append(current)
        current = came_from[current]

    #Add the start to the path.
    path.append(start)
    #Reverse the path list so it can be followed along as start -> goal, rather than goal -> start which it was.
    path.reverse()
    #Return the newly created path.
    return path

## Breadth-First Search algorithm

In [29]:
def breadth_first_search(grid, start, goal, rows, columns, car_id):
    #Setup the frontier as a regular queue.
    frontier = deque([start])
    came_from = {}
    came_from[start] = None

    #While there are still grid spaces to explore, continue searching.
    while frontier:
        current = frontier.popleft()

        #Check if the goal has been reached.
        if current == goal:
            return reconstruct_path(came_from, start, goal)

        #Explore each valid neighbour.
        for neighbour in get_neighbours(current, rows, columns):
            #If the neighbour is occupied or invalid, skip.
            if grid[neighbour[0], neighbour[1]] != 0 and grid[neighbour[0], neighbour[1]] != car_id and neighbour != goal:
                continue

            #Ensure that each node is only visited once.
            if neighbour not in came_from:
                frontier.append(neighbour)
                came_from[neighbour] = current

    #Return an empty list if no path is found.
    return []

## A Star algorithm

In [30]:
#A Star algorithm
def a_star(grid, start, goal, rows, columns, car_id):
    #Setup the frontier as a priority queue. A priority queue is a queue where each item 
    #in the queue can be given a different priority, meaning that higher priority items are 
    #dequeued first.
    frontier = PriorityQueue()
    #Add the start to the queue, give it a priority of 0.
    frontier.put((0, start))

    #Setup dictionaries to contain came from and cost so far.
    came_from = {}
    cost_so_far = {}

    #Initialise their start position to be none (no parent yet) and 0 (no cost yet).
    came_from[start] = None
    cost_so_far[start] = 0

    #While the frontier is not empty, continue to search.
    while not frontier.empty():
        #Only get the current value from the queue, as we don't need the priority value, discard it.
        _, current = frontier.get()

        #Check if the goal has been reached.
        if current == goal:
            return reconstruct_path(came_from, start, goal)

        #Explore each valid neighbour of the current position of the frontier.
        for neighbour in get_neighbours(current, rows, columns):
            #Skip if the neighbour is occupied by a car or obstacle, unless it's the goal.
            if grid[neighbour[0], neighbour[1]] != 0 and grid[neighbour[0], neighbour[1]] != car_id and neighbour != goal:
                continue

            #Calculate the new cost, each move costs 1.
            new_cost = cost_so_far[current] + 1

            #If this is a new position or a cheaper path to this position has been found, update the frontier. 
            if neighbour not in cost_so_far or new_cost < cost_so_far[neighbour]:
                cost_so_far[neighbour] = new_cost
                #The priority is the actual cost added to the estimated cost to the goal.
                priority = new_cost + manhattan_distance(neighbour, goal)
                frontier.put((priority, neighbour))
                came_from[neighbour] = current
    #Return an empty list if no path could be found.
    return []

## Update simulation

In [31]:
def update_simulation(grid, cars, rows, columns, algorithm, max_wait = 3):

    algorithms = {
        'A*': a_star,
        'BFS': breadth_first_search
    }

    for car in cars.values():
        
        #If current car has reached goal, skip to the next car.
        if car.reached_goal:
            continue

        #If the car doesn't have a path, calculate one using the A* algorithm.
        if not car.path:
            car.path = algorithms[algorithm](grid, car.position, car.goal, rows, columns, car.id)

        #If there is still path remaining for the car to traverse, attempt to move to the next step.
        if len(car.path) > 1:
            car.steps_taken += 1
            #Set next position to the next step along the path.
            next_position = car.path[1]

            #If the next position is free, move the car to that position, remove the current position from the path, reset wait counter to zero and increment total distance and steps taken by 1.
            if grid[next_position[0], next_position[1]] == 0:
                grid[car.position[0], car.position[1]] = 0
                car.position = next_position
                grid[next_position[0], next_position[1]] = car.id
                car.path.pop(0)
                car.wait_counter = 0
                car.total_distance_travelled += 1
                

                #If the car's new position is its goal, print a message and set its reached_goal boolean to True.
                if car.position == car.goal:
                    print(f"Car {car.id} has reached goal.")
                    car.reached_goal = True

            #If the next space is occupied by another car/object, increment the wait counter by 1.
            else:
                car.wait_counter += 1

                #If the space is still occupied after the max amount of waits has been reached, recalculate the path by running the A* algorithm again.
                if car.wait_counter >= max_wait:
                    print(f"Car {car.id} replanning route after waiting {car.wait_counter} steps.")
                    car.path = algorithms[algorithm](grid, car.position, car.goal, rows, columns, car.id)
                    car.wait_counter = 0

## Create GIF

In [32]:
#Create a snapshot of the current state of the grid.
def create_snapshot(grid, cars, rows, columns, timestep, cell_size=20):

    # Calculate frame dimensions.
    image_width = columns * cell_size
    image_height = rows * cell_size + 50  
    
    # Create frame.
    image = Image.new('RGB', (image_width, image_height), color='white')
    draw = ImageDraw.Draw(image)
    
    # Draw cells on grid.
    for row in range(rows):
        for column in range(columns):
            x1 = column * cell_size
            y1 = row * cell_size + 50  
            x2 = x1 + cell_size
            y2 = y1 + cell_size
            
            cell_value = grid[row, column]
            
            if cell_value == 0:
                # Empty cell = light grey.
                colour = (240, 240, 240)
            else:
                # Different colour for each car.
                colours = [
                    #Red
                    (255, 100, 100), 
                    #Blue
                    (0, 255, 255), 
                    #Green
                    (100, 255, 100), 
                    #Yellow
                    (255, 255, 100), 
                    #Pink
                    (255, 100, 255),  
                ]
                #Decide the colour of the given cell.
                colour = colours[(cell_value - 1) % len(colours)]
            
            # Draw cell.
            draw.rectangle([x1, y1, x2, y2], fill=colour, outline='black', width=1)
            
            # Draw car ID.
            if cell_value > 0:
                text_x = x1 + cell_size // 2
                text_y = y1 + cell_size // 2
                draw.text((text_x, text_y), str(cell_value), fill='black', anchor='mm')
    
    # Draw goals as circles.
    for car in cars.values():
        goal_row, goal_column = car.goal
        x = goal_column * cell_size + cell_size // 2
        y = goal_row * cell_size + cell_size // 2 + 50
        radius = cell_size // 4
        
        # Draw goal marker.
        draw.ellipse(
            [x - radius, y - radius, x + radius, y + radius],
            outline='red',
            width=3
        )
    
    # Draw timestep information.
    cars_at_goal = sum(1 for c in cars.values() if c.reached_goal)
    draw.text((10, 10), f"Timestep: {timestep}", fill='black')
    draw.text((10, 30), f"Cars at goal: {cars_at_goal}/{len(cars)}", fill='black')
    
    return image

In [33]:
#Save all frames as a GIF.
def save_frames_as_gif(frames, filename='traffic_simulation.gif', duration=200):
    if frames:
        frames[0].save(
            filename,
            save_all=True,
            append_images=frames[1:],
            duration=duration,
            loop=0
        )
        print(f"GIF saved as {filename}")

## Run Simulation

In [34]:
#Run sim and create a gif
def run_simulation(num_cars, rows, columns, algorithm, max_timesteps=200, max_wait=3):
    
    frames = []
    grid, cars = setup_grid(num_cars, rows, columns)
    print("Running simulation")
    
    for timestep in range(max_timesteps):
        # Capture frame before updating.
        frame = create_snapshot(grid, cars, rows, columns, timestep)
        frames.append(frame)
        
        # Update simulation.
        update_simulation(grid, cars, rows, columns, algorithm, max_wait)
        
        # Print progress every 20 steps.
        if timestep % 20 == 0:
            cars_at_goal = sum(1 for c in cars.values() if c.reached_goal)
            print(f"Timestep {timestep}: {cars_at_goal}/{len(cars)} cars at goal.")
        
        # Check if all cars have reached their goals.
        if all(car.reached_goal for car in cars.values()):
            print(f"\nAll cars reached goals at timestep {timestep}.")
            
            # Add 5 seconds of freeze frames at the end
            final_frame = create_snapshot(grid, cars, rows, columns, timestep)
            #Append the final frame, otherwise it won't appear in the GIF.
            frames.append(final_frame)

            #Adds 50 copies of the final frame to add the 5 secs of freeze frame to the end.
            for _ in range(50):
                frames.append(final_frame)
            break
    
    # If simulation didn't complete, still freeze at the end.
    else:
        final_frame = create_snapshot(grid, cars, rows, columns, max_timesteps - 1)
        for _ in range(50):
            frames.append(final_frame)
    
    # Save GIF.
    save_frames_as_gif(frames, 'traffic_simulation.gif', duration=100)
    
    # Print final statistics.
    print("\n=== Final Statistics ===")
    for car in cars.values():
        if car.reached_goal:
            print(f"Car {car.id}: Steps taken = {car.steps_taken}, Distance = {car.total_distance_travelled}.")
        else:
            print(f"Car {car.id}: Did NOT reach goal.")
    
    return frames

In [35]:
# Run the simulation
frames = run_simulation(num_cars = 50, rows = 30, columns = 30, algorithm = 'BFS', max_timesteps=200, max_wait=3)

Running simulation
Car 30 has reached goal.
Timestep 0: 1/50 cars at goal.
Car 20 has reached goal.
Car 45 has reached goal.
Car 2 has reached goal.
Car 28 has reached goal.
Car 46 has reached goal.
Car 41 has reached goal.
Car 50 has reached goal.
Car 10 replanning route after waiting 3 steps.
Car 13 replanning route after waiting 3 steps.
Car 34 replanning route after waiting 3 steps.
Car 47 replanning route after waiting 3 steps.
Car 14 has reached goal.
Car 43 has reached goal.
Car 11 replanning route after waiting 3 steps.
Car 29 replanning route after waiting 3 steps.
Car 6 replanning route after waiting 3 steps.
Car 33 replanning route after waiting 3 steps.
Car 3 has reached goal.
Car 12 replanning route after waiting 3 steps.
Car 23 has reached goal.
Car 24 replanning route after waiting 3 steps.
Car 25 replanning route after waiting 3 steps.
Car 34 replanning route after waiting 3 steps.
Car 48 replanning route after waiting 3 steps.
Car 1 replanning route after waiting 3 ste